# Chunking and Retrieval Evaluation

This notebook compares chunking and retrieval strategies using predefined development questions.

Before creating chunks, we validate that every gold evidence page exists and is eligible for retrieval. This prevents us from evaluating retrieval against incorrect references.

In [ ]:
import json
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

questions_path = PROJECT_ROOT / "evaluation" / "development_questions.json"
pages_path = PROJECT_ROOT / "data" / "processed" / "pages.jsonl"

print("Project root:", PROJECT_ROOT)
print("Questions path:", questions_path)

questions = json.loads(
    questions_path.read_text(encoding="utf-8")
)

pages = pd.read_json(pages_path, lines=True)


from aviation_rag.evaluation_data import validate_gold_evidence

validation_errors = validate_gold_evidence(questions, pages)

if validation_errors:
    raise ValueError("\n".join(validation_errors))

print("All development-question evidence passed validation.")
print("Validated questions:", len(questions))

### Validation Interpretation

All 12 development questions passed the evidence check. This does not mean retrieval answered them correctly. It only confirms that every answerable question points to existing, retrieval-eligible gold pages and that refusal questions do not incorrectly contain gold evidence.

This validation protects the evaluation itself. Without it, a retriever could be marked wrong because the answer key contains an invalid page.\n

## Baseline 1: Whole-Page Chunks

The first baseline treats each retrieval-eligible PDF page as one chunk.

This strategy is simple and preserves page boundaries, making citations straightforward. However, a long page may contain several topics. Its embedding may therefore represent the page too broadly and reduce retrieval precision.

We use this as a baseline, not as the assumed final strategy.

In [ ]:
page_chunks = pages.loc[pages["retrieval_eligible"]].copy()

page_chunks["chunk_id"] = (
    page_chunks["document_id"]
    + "::page_"
    + page_chunks["page_number"].astype(str).str.zfill(3)
)

page_chunks["chunk_text"] = page_chunks["text"]
page_chunks["chunk_strategy"] = "whole_page"
page_chunks["character_count"] = page_chunks["chunk_text"].str.len()

assert page_chunks["chunk_id"].is_unique
assert page_chunks["chunk_text"].str.strip().ne("").all()
#total pages ada 186 (201 - 15 = 186 whole-page chunks)
assert len(page_chunks) == 186

print("Whole-page chunks:", len(page_chunks))
print(
    page_chunks["character_count"]
    .describe()[["min", "25%", "50%", "75%", "max"]]
    .round(1)
)

### Whole-Page Chunk Interpretation

The 201 source pages contain 186 retrieval-eligible pages after 15 covers, dividers, image-dependent tables, and other non-retrieval pages were excluded. Whole-page chunking creates exactly one chunk for each eligible page, so it produces 186 chunks.

Page length varies considerably, from 278 to 3,782 characters. This variation is a warning: long pages may mix several topics or exceed an embedding model's token limit, while short pages may represent one focused fact. The count alone does not prove that whole-page chunking is suitable.\n

## Baseline 2: Word-Window Chunks

This strategy divides each eligible page into chunks containing at most 180 words.

Adjacent chunks overlap by 30 words. The overlap reduces the risk of separating a rule from its surrounding explanation. Chunks never cross page boundaries, so every retrieved chunk still has a clear PDF page citation.

This remains a simple baseline. It respects words, but it may still split sentences or legal provisions.

In [ ]:
from aviation_rag.chunking import build_word_window_chunks


word_chunks = build_word_window_chunks(
    pages=pages,
    max_words=180,
    overlap_words=30,
)

print("Word-window chunks:", len(word_chunks))
print(
    word_chunks["word_count"]
    .describe()[["min", "25%", "50%", "75%", "max"]]
    .round(1)
)

### Interpretation

The 186 eligible pages produced 307 word-window chunks because longer pages were divided into multiple overlapping units.

The smaller chunks may represent individual facts more clearly than whole pages. However, they may separate related sentences or legal provisions, while overlap may produce repetitive retrieval results. Therefore, chunk size cannot be selected from the length distribution alone. Both strategies must be tested using the same development questions and retrieval metric.

## TF-IDF Retrieval Baseline

TF-IDF represents questions and chunks using weighted word and phrase frequencies. Cosine similarity then ranks chunks by their lexical similarity to the question.

This is not semantic retrieval. It may fail when the question and evidence use different terminology. It provides a cheap, reproducible baseline that the later embedding retriever must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


whole_page_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

whole_page_matrix = whole_page_vectorizer.fit_transform(
    page_chunks["chunk_text"]
)

print("Chunk matrix shape:", whole_page_matrix.shape)

### TF-IDF Matrix Interpretation

The matrix shape `(186, 18314)` means there are 186 whole-page chunks and 18,314 learned lexical features. These features are unigrams and bigrams, not 18,314 manually selected columns.

Each row represents one chunk. Each column represents a word or two-word phrase learned from the corpus. Most values are zero because any single page uses only a small part of the full vocabulary.\n

In [ ]:
def retrieve_tfidf(
    query: str,
    chunks: pd.DataFrame,
    vectorizer: TfidfVectorizer,
    chunk_matrix,
    top_k: int = 5,
) -> pd.DataFrame:
    """Return the highest-scoring chunks for a query."""

    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, chunk_matrix).ravel()
    top_indices = scores.argsort()[::-1][:top_k]

    results = chunks.iloc[top_indices][
        ["chunk_id", "document_id", "page_number", "chunk_text"]
    ].copy()

    results["score"] = scores[top_indices]

    return results.reset_index(drop=True)

In [ ]:
sample_question = questions[0]["question"]

print("Question:", sample_question)

sample_results = retrieve_tfidf(
    query=sample_question,
    chunks=page_chunks,
    vectorizer=whole_page_vectorizer,
    chunk_matrix=whole_page_matrix,
    top_k=5,
)

print(
    sample_results[
        ["document_id", "page_number", "score"]
    ].to_string(index=False)
)

### Sample TF-IDF Ranking Interpretation

The correct principal-Code page appears at rank 3, so this question is a Hit@3 and Hit@5 success but a Hit@1 failure. The two amendment commencement pages rank above it because they share strong exact terms such as the Code title, citation, and commencement language.

This example shows both the strength and limitation of TF-IDF. Exact legal wording is useful, but several legally related documents can contain nearly identical vocabulary. Retrieval rank must therefore be evaluated rather than judged from one plausible result.\n

## Whole-Page TF-IDF Evaluation

Retrieval is evaluated only on answerable questions.

Recall@K measures the proportion of required gold pages found within the top K retrieved chunks. Hit@K records whether at least one gold page was found. For questions with one gold page, Recall@K and Hit@K are identical. They differ for questions requiring evidence from multiple pages.

In [ ]:
def evaluate_retrieval(
    questions: list[dict],
    chunks: pd.DataFrame,
    vectorizer: TfidfVectorizer,
    chunk_matrix,
) -> pd.DataFrame:
    """Evaluate page-level gold evidence at K values 1, 3, and 5."""

    evaluation_records = []

    for item in questions:
        if not item["answerable"]:
            continue

        gold_pages = {
            (document_id, int(page_number))
            for document_id, page_numbers in item["gold_pages"].items()
            for page_number in page_numbers
        }

        retrieved = retrieve_tfidf(
            query=item["question"],
            chunks=chunks,
            vectorizer=vectorizer,
            chunk_matrix=chunk_matrix,
            top_k=5,
        )

        retrieved_pages = list(
            zip(
                retrieved["document_id"],
                retrieved["page_number"].astype(int),
            )
        )

        record = {
            "question_id": item["question_id"],
            "category": item["category"],
            "gold_page_count": len(gold_pages),
        }

        for k in (1, 3, 5):
            top_k_pages = set(retrieved_pages[:k])
            matched_pages = gold_pages & top_k_pages

            record[f"recall_at_{k}"] = (
                len(matched_pages) / len(gold_pages)
            )
            record[f"hit_at_{k}"] = int(bool(matched_pages))

        evaluation_records.append(record)

    return pd.DataFrame(evaluation_records)

In [ ]:
whole_page_evaluation = evaluate_retrieval(
    questions=questions,
    chunks=page_chunks,
    vectorizer=whole_page_vectorizer,
    chunk_matrix=whole_page_matrix,
)

metric_columns = [
    "recall_at_1",
    "recall_at_3",
    "recall_at_5",
    "hit_at_1",
    "hit_at_3",
    "hit_at_5",
]

print("Answerable questions:", len(whole_page_evaluation))
print("\nMean metrics:")
print(whole_page_evaluation[metric_columns].mean().round(3))

print("\nQuestions without full Recall@5:")
print(
    whole_page_evaluation.loc[
        whole_page_evaluation["recall_at_5"] < 1,
        ["question_id", "category", "gold_page_count", "recall_at_5", "hit_at_5"],
    ].to_string(index=False)
)

### Interpretation

The whole-page TF-IDF baseline achieved mean Recall@5 of 0.778. Seven of the nine answerable development questions retrieved all required gold pages, while two retrieved none.

Recall@3 and Recall@5 were identical, showing that ranks four and five added no gold evidence. The baseline failed on one report fact question and the cross-document synthesis question. The synthesis failure is especially important because none of its three required pages appeared in the top five, leaving an answer generator without the evidence needed for a grounded response.

These results are diagnostic only. The development set is small and is used for improving the system, not for making final performance claims.

In [ ]:
failed_question_ids = ["DEV_REPORT_001", "DEV_SYNTH_001"]

for question_id in failed_question_ids:
    item = next(
        question
        for question in questions
        if question["question_id"] == question_id
    )

    results = retrieve_tfidf(
        query=item["question"],
        chunks=page_chunks,
        vectorizer=whole_page_vectorizer,
        chunk_matrix=whole_page_matrix,
        top_k=5,
    )

    print("=" * 80)
    print("Question ID:", question_id)
    print("Question:", item["question"])
    print("Gold pages:", item["gold_pages"])
    print("\nRetrieved pages:")

    print(
        results[
            ["document_id", "page_number", "score"]
        ].to_string(index=False)
    )

In [ ]:
word_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

word_chunk_matrix = word_vectorizer.fit_transform(
    word_chunks["chunk_text"]
)

word_chunk_evaluation = evaluate_retrieval(
    questions=questions,
    chunks=word_chunks,
    vectorizer=word_vectorizer,
    chunk_matrix=word_chunk_matrix,
)

print("Chunk matrix shape:", word_chunk_matrix.shape)

print("\nMean metrics:")
print(
    word_chunk_evaluation[metric_columns]
    .mean()
    .round(3)
)

print("\nQuestions without full Recall@5:")
print(
    word_chunk_evaluation.loc[
        word_chunk_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

In [ ]:
strategy_comparison = whole_page_evaluation.merge(
    word_chunk_evaluation,
    on=["question_id", "category", "gold_page_count"],
    suffixes=("_whole_page", "_word_window"),
)

difference_columns = []

for metric in metric_columns:
    difference_column = f"{metric}_difference"
    difference_columns.append(difference_column)

    strategy_comparison[difference_column] = (
        strategy_comparison[f"{metric}_word_window"]
        - strategy_comparison[f"{metric}_whole_page"]
    )

changed_questions = strategy_comparison.loc[
    strategy_comparison[difference_columns].abs().sum(axis=1) > 0
]

if changed_questions.empty:
    print("No question-level metric differences were found.")
else:
    print(
        changed_questions[
            ["question_id", *difference_columns]
        ].to_string(index=False)
    )

### Chunking Comparison Interpretation

The whole-page and 180-word window strategies produced identical question-level TF-IDF metrics. The smaller chunks did not improve or worsen Recall@K or Hit@K for any development question.

This result suggests that the observed failures are not solved merely by reducing chunk size. Vocabulary mismatch and multi-intent questions remain likely causes. However, this conclusion applies only to the current development questions and TF-IDF retriever. The two chunking strategies must also be compared using semantic embeddings before selecting one.

## Semantic Embedding Retrieval

A semantic embedding converts text into a dense numerical vector. Texts with similar meanings should have nearby vectors even when they use different words.

The embedding model is pretrained. We are using it for inference only, which means we are not training it or applying backpropagation.

We begin with `all-MiniLM-L6-v2` as the lightweight semantic baseline. Model selection will be based on measured retrieval performance, not popularity.

In [ ]:
from sentence_transformers import SentenceTransformer


MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

minilm_model = SentenceTransformer(MINILM_MODEL_NAME)

print("Model:", MINILM_MODEL_NAME)
print("Embedding dimensions:", minilm_model.get_embedding_dimension())
print("Maximum sequence length:", minilm_model.max_seq_length)

### Token-Length Validation

The MiniLM model accepts a maximum of 256 tokens. Text beyond this limit is truncated during embedding.

We measure token lengths before encoding because character and word counts do not exactly match model tokens. A chunking strategy that regularly exceeds the model limit is not acceptable, even if its retrieval metrics appear reasonable.

In [ ]:
def calculate_token_lengths(
    chunks: pd.DataFrame,
    model: SentenceTransformer,
) -> pd.Series:
    """Calculate token counts without truncating the text."""

    tokenized = model.tokenizer(
        chunks["chunk_text"].tolist(),
        add_special_tokens=True,
        truncation=False,
        padding=False,
    )

    return pd.Series(
        [len(token_ids) for token_ids in tokenized["input_ids"]],
        index=chunks.index,
    )


page_chunks["token_count_minilm"] = calculate_token_lengths(
    page_chunks,
    minilm_model,
)

word_chunks["token_count_minilm"] = calculate_token_lengths(
    word_chunks,
    minilm_model,
)

for strategy_name, chunks in [
    ("whole_page", page_chunks),
    ("word_window", word_chunks),
]:
    token_counts = chunks["token_count_minilm"]
    over_limit = (token_counts > minilm_model.max_seq_length).sum()

    print(f"\nStrategy: {strategy_name}")
    print(f"Chunks: {len(chunks)}")
    print(f"Chunks over model limit: {over_limit}")
    print(f"Percentage over limit: {over_limit / len(chunks):.1%}")

### Token-Length Interpretation

Whole-page chunking is unsafe for MiniLM because 51.6% of page chunks exceed the model's 256-token limit. Text after the limit would be truncated, so evidence near the end of many pages would not influence their embeddings.

The 180-word strategy reduces this problem substantially, but 25 chunks still exceed the limit. Even a small number of truncated chunks may contain important regulatory evidence. The word window must therefore be reduced and validated again.

In [ ]:
revised_word_chunks = build_word_window_chunks(
    pages=pages,
    max_words=140,
    overlap_words=25,
)

revised_word_chunks["token_count_minilm"] = (
    calculate_token_lengths(
        revised_word_chunks,
        minilm_model,
    )
)

token_counts = revised_word_chunks["token_count_minilm"]

print("Revised chunks:", len(revised_word_chunks))
print("Maximum tokens:", token_counts.max())
print(
    "Chunks over model limit:",
    (
        token_counts
        > minilm_model.max_seq_length
    ).sum(),
)

### Revised Chunk Interpretation

Reducing the window to 140 words with 25 words of overlap produced 368 chunks. The maximum MiniLM token length fell to 246, below the model limit of 256, so no chunk will be silently truncated during embedding.

This is a technical safety improvement, not proof of better retrieval. The revised chunks still need evaluation because a token-safe chunk can remain semantically incomplete or split evidence across neighbouring chunks.\n

In [ ]:
minilm_chunk_embeddings = minilm_model.encode(
    revised_word_chunks["chunk_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embedding matrix shape:", minilm_chunk_embeddings.shape)

In [ ]:
def retrieve_semantic(
    query: str,
    chunks: pd.DataFrame,
    model: SentenceTransformer,
    chunk_embeddings,
    top_k: int = 5,
) -> pd.DataFrame:
    """Return chunks ranked by semantic cosine similarity."""

    query_embedding = model.encode_query(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    scores = chunk_embeddings @ query_embedding
    top_indices = scores.argsort()[::-1][:top_k]

    results = chunks.iloc[top_indices][
        ["chunk_id", "document_id", "page_number", "chunk_text"]
    ].copy()

    results["score"] = scores[top_indices]

    return results.reset_index(drop=True)

In [ ]:
sample_semantic_results = retrieve_semantic(
    query=questions[0]["question"],
    chunks=revised_word_chunks,
    model=minilm_model,
    chunk_embeddings=minilm_chunk_embeddings,
    top_k=5,
)

print("Question:", questions[0]["question"])
print(
    sample_semantic_results[
        ["document_id", "page_number", "score"]
    ].to_string(index=False)
)

In [ ]:
#why cant find answer
print("Embedding matrix shape:", minilm_chunk_embeddings.shape)

sample_query_embedding = minilm_model.encode(
    [questions[0]["question"]],
    convert_to_numpy=True,
    normalize_embeddings=True,
)[0]

sample_scores = minilm_chunk_embeddings @ sample_query_embedding

rank_diagnostics = revised_word_chunks[
    ["chunk_id", "document_id", "page_number", "chunk_index", "chunk_text"]
].copy()

rank_diagnostics["score"] = sample_scores
rank_diagnostics["rank"] = (
    rank_diagnostics["score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

gold_page_chunks = rank_diagnostics.loc[
    (rank_diagnostics["document_id"] == "macpc_principal_2016")
    & (rank_diagnostics["page_number"] == 39)
].sort_values("rank")

print(
    gold_page_chunks[
        ["chunk_index", "rank", "score"]
    ].to_string(index=False)
)

print("\nBest gold chunk text:")
print(gold_page_chunks.iloc[0]["chunk_text"])

### MiniLM Sample-Rank Interpretation

The answer-bearing chunk from principal-Code page 39 ranked 6, just outside the evaluated top five. Another chunk from the same gold page ranked 365 because it contains different text and little information relevant to the question.

This demonstrates why a gold page can contain both useful and irrelevant chunks. Page-level evaluation counts the page as found when any retrieved chunk from that page appears, but the retriever still ranks individual chunks based on their own contents.\n

In [ ]:
def evaluate_semantic_retrieval(
    questions: list[dict],
    chunks: pd.DataFrame,
    model: SentenceTransformer,
    chunk_embeddings,
) -> pd.DataFrame:
    """Evaluate semantic retrieval against page-level gold evidence."""

    evaluation_records = []

    for item in questions:
        if not item["answerable"]:
            continue

        gold_pages = {
            (document_id, int(page_number))
            for document_id, page_numbers in item["gold_pages"].items()
            for page_number in page_numbers
        }

        retrieved = retrieve_semantic(
            query=item["question"],
            chunks=chunks,
            model=model,
            chunk_embeddings=chunk_embeddings,
            top_k=5,
        )

        retrieved_pages = list(
            zip(
                retrieved["document_id"],
                retrieved["page_number"].astype(int),
            )
        )

        record = {
            "question_id": item["question_id"],
            "category": item["category"],
            "gold_page_count": len(gold_pages),
        }

        for k in (1, 3, 5):
            top_k_pages = set(retrieved_pages[:k])
            matched_pages = gold_pages & top_k_pages

            record[f"recall_at_{k}"] = (
                len(matched_pages) / len(gold_pages)
            )
            record[f"hit_at_{k}"] = int(bool(matched_pages))

        evaluation_records.append(record)

    return pd.DataFrame(evaluation_records)

In [ ]:
minilm_evaluation = evaluate_semantic_retrieval(
    questions=questions,
    chunks=revised_word_chunks,
    model=minilm_model,
    chunk_embeddings=minilm_chunk_embeddings,
)

print("Mean metrics:")
print(minilm_evaluation[metric_columns].mean().round(3))

print("\nQuestions without full Recall@5:")
print(
    minilm_evaluation.loc[
        minilm_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

### MiniLM Evaluation Interpretation

MiniLM achieved mean Recall@5 of 0.278, substantially below the TF-IDF baseline of 0.778. It failed to retrieve any gold evidence for several regulatory, report, and synthesis questions.

This demonstrates that semantic embeddings are not automatically better than lexical retrieval. Exact legal titles, dates, amendment numbers, and reporting-period terminology can favour lexical matching. MiniLM should not be selected merely because it uses a neural model.

The result applies to this model and current configuration. It does not prove that all embedding models will perform poorly.

In [ ]:
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"

bge_model = SentenceTransformer(BGE_MODEL_NAME)

print("Model:", BGE_MODEL_NAME)
print("Embedding dimensions:", bge_model.get_embedding_dimension())
print("Maximum sequence length:", bge_model.max_seq_length)
print("Available prompts:", bge_model.prompts)
print("Default prompt:", bge_model.default_prompt_name)

In [ ]:
revised_word_chunks["token_count_bge"] = calculate_token_lengths(
    revised_word_chunks,
    bge_model,
)

bge_token_counts = revised_word_chunks["token_count_bge"]

print("Maximum BGE tokens:", bge_token_counts.max())
print(
    "Chunks over BGE limit:",
    (bge_token_counts > bge_model.max_seq_length).sum(),
)

In [ ]:
bge_chunk_embeddings = bge_model.encode_document(
    revised_word_chunks["chunk_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("BGE embedding matrix shape:", bge_chunk_embeddings.shape)

### BGE Setup Interpretation

BGE creates one 384-dimensional vector for each of the 368 revised chunks, producing a `(368, 384)` embedding matrix. The maximum chunk length is 246 tokens, safely below BGE's 512-token limit.

BGE's larger token limit does not justify returning to whole-page chunks automatically. Chunk size also affects topic focus, evidence completeness, citation precision, and retrieval competition.\n

In [ ]:
bge_evaluation = evaluate_semantic_retrieval(
    questions=questions,
    chunks=revised_word_chunks,
    model=bge_model,
    chunk_embeddings=bge_chunk_embeddings,
)

print("Mean metrics:")
print(bge_evaluation[metric_columns].mean().round(3))

print("\nQuestions without full Recall@5:")
print(
    bge_evaluation.loc[
        bge_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

### BGE Evaluation Interpretation

BGE achieved Recall@5 of 0.648, improving substantially over MiniLM's 0.278 but remaining below TF-IDF's 0.778.

BGE and TF-IDF both achieved Hit@5 of 0.778, meaning both retrieved at least one gold page for the same proportion of questions. BGE's lower Recall@5 shows that it retrieved fewer of the total required pages for multi-page questions.

Therefore, BGE improves semantic retrieval over MiniLM but has not yet beaten the lexical baseline.

In [ ]:
BGE_QUERY_INSTRUCTION = (
    "Represent this sentence for searching relevant passages: "
)

bge_instructed_questions = [
    {
        **item,
        "question": BGE_QUERY_INSTRUCTION + item["question"],
    }
    for item in questions
]

bge_instructed_evaluation = evaluate_semantic_retrieval(
    questions=bge_instructed_questions,
    chunks=revised_word_chunks,
    model=bge_model,
    chunk_embeddings=bge_chunk_embeddings,
)

print("Mean metrics:")
print(
    bge_instructed_evaluation[metric_columns]
    .mean()
    .round(3)
)

print("\nQuestions without full Recall@5:")
print(
    bge_instructed_evaluation.loc[
        bge_instructed_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

In [ ]:
word_chunks["token_count_bge"] = calculate_token_lengths(
    word_chunks,
    bge_model,
)

print("Maximum BGE tokens:", word_chunks["token_count_bge"].max())
print(
    "Chunks over BGE limit:",
    (
        word_chunks["token_count_bge"]
        > bge_model.max_seq_length
    ).sum(),
)

In [ ]:
bge_180_chunk_embeddings = bge_model.encode_document(
    word_chunks["chunk_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

bge_180_instructed_evaluation = evaluate_semantic_retrieval(
    questions=bge_instructed_questions,
    chunks=word_chunks,
    model=bge_model,
    chunk_embeddings=bge_180_chunk_embeddings,
)

print("Mean metrics:")
print(
    bge_180_instructed_evaluation[metric_columns]
    .mean()
    .round(3)
)

print("\nQuestions without full Recall@5:")
print(
    bge_180_instructed_evaluation.loc[
        bge_180_instructed_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

In [ ]:
tfidf_140_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True,
)

tfidf_140_matrix = tfidf_140_vectorizer.fit_transform(
    revised_word_chunks["chunk_text"]
)

tfidf_140_evaluation = evaluate_retrieval(
    questions=questions,
    chunks=revised_word_chunks,
    vectorizer=tfidf_140_vectorizer,
    chunk_matrix=tfidf_140_matrix,
)

print("Mean metrics:")
print(
    tfidf_140_evaluation[metric_columns]
    .mean()
    .round(3)
)

print("\nQuestions without full Recall@5:")
print(
    tfidf_140_evaluation.loc[
        tfidf_140_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

In [ ]:
experiment_results = {
    "TF-IDF | whole page": whole_page_evaluation,
    "TF-IDF | 180 words": word_chunk_evaluation,
    "TF-IDF | 140 words": tfidf_140_evaluation,
    "MiniLM | 140 words": minilm_evaluation,
    "BGE | 140 words | no instruction": bge_evaluation,
    "BGE | 140 words | instruction": bge_instructed_evaluation,
    "BGE | 180 words | instruction": bge_180_instructed_evaluation,
}

summary_records = []

for experiment_name, results in experiment_results.items():
    mean_metrics = results[metric_columns].mean()

    summary_records.append(
        {
            "experiment": experiment_name,
            **{
                metric: mean_metrics[metric]
                for metric in metric_columns
            },
        }
    )

experiment_summary = (
    pd.DataFrame(summary_records)
    .sort_values(
        ["recall_at_5", "hit_at_5"],
        ascending=False,
    )
    .reset_index(drop=True)
)

print(experiment_summary.round(3).to_string(index=False))

In [ ]:
per_question_comparison = (
    tfidf_140_evaluation[
        [
            "question_id",
            "category",
            "recall_at_5",
            "hit_at_5",
        ]
    ]
    .merge(
        bge_instructed_evaluation[
            [
                "question_id",
                "recall_at_5",
                "hit_at_5",
            ]
        ],
        on="question_id",
        suffixes=("_tfidf", "_bge"),
    )
)

per_question_comparison["recall_difference_bge_minus_tfidf"] = (
    per_question_comparison["recall_at_5_bge"]
    - per_question_comparison["recall_at_5_tfidf"]
)

print(
    per_question_comparison[
        [
            "question_id",
            "category",
            "recall_at_5_tfidf",
            "recall_at_5_bge",
            "hit_at_5_tfidf",
            "hit_at_5_bge",
            "recall_difference_bge_minus_tfidf",
        ]
    ].to_string(index=False)
)

### Retrieval Experiment Comparison

TF-IDF achieved the strongest individual Recall@5 at 0.778 across all three tested chunk sizes. The identical TF-IDF metrics show that changing from whole pages to 180-word or 140-word windows did not change page-level success on this small development set.

MiniLM performed substantially worse. BGE improved over MiniLM, and its query instruction raised Recall@5 from 0.648 to 0.704, but it remained below TF-IDF. The 180-word BGE configuration performed worse than the 140-word configuration even though neither exceeded BGE's token limit. This shows that a larger valid chunk is not automatically a better retrieval unit.

The per-question comparison reveals complementary behaviour. TF-IDF found an amendment page that BGE missed, while BGE found part of the synthesis evidence that TF-IDF missed. That evidence supports testing a hybrid retriever instead of replacing TF-IDF with the more complex model.\n

## Hybrid Retrieval with Reciprocal Rank Fusion

TF-IDF and BGE show complementary retrieval behaviour. TF-IDF retrieved evidence that BGE missed for one amendment question, while BGE retrieved partial evidence for the synthesis question that TF-IDF missed entirely.

Reciprocal Rank Fusion combines their ranked results. A chunk receives points based on its rank in each retriever. Chunks ranked highly by both systems receive stronger combined scores.

RRF uses ranking positions rather than raw similarity scores because TF-IDF and BGE scores are not directly comparable.

In [ ]:
def retrieve_hybrid_rrf(
    query: str,
    top_k: int = 5,
    candidate_k: int = 20,
    rrf_constant: int = 60,
    tfidf_weight: float = 0.5,
) -> pd.DataFrame:
    """Combine TF-IDF and BGE rankings using reciprocal rank fusion."""

    if not 0 <= tfidf_weight <= 1:
        raise ValueError("tfidf_weight must be between 0 and 1")

    bge_weight = 1 - tfidf_weight

    tfidf_results = retrieve_tfidf(
        query=query,
        chunks=revised_word_chunks,
        vectorizer=tfidf_140_vectorizer,
        chunk_matrix=tfidf_140_matrix,
        top_k=candidate_k,
    )

    bge_results = retrieve_semantic(
        query=BGE_QUERY_INSTRUCTION + query,
        chunks=revised_word_chunks,
        model=bge_model,
        chunk_embeddings=bge_chunk_embeddings,
        top_k=candidate_k,
    )

    fused_scores = {}

    for results, weight in [
        (tfidf_results, tfidf_weight),
        (bge_results, bge_weight),
    ]:
        for rank, chunk_id in enumerate(
            results["chunk_id"],
            start=1,
        ):
            fused_scores[chunk_id] = (
                fused_scores.get(chunk_id, 0.0)
                + weight / (rrf_constant + rank)
            )

    candidate_ids = set(fused_scores)

    fused_results = revised_word_chunks.loc[
        revised_word_chunks["chunk_id"].isin(candidate_ids),
        ["chunk_id", "document_id", "page_number", "chunk_text"],
    ].copy()

    fused_results["score"] = (
        fused_results["chunk_id"].map(fused_scores)
    )

    return (
        fused_results
        .sort_values("score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
for question_id in ["DEV_REPORT_001", "DEV_SYNTH_001"]:
    item = next(
        question
        for question in questions
        if question["question_id"] == question_id
    )

    results = retrieve_hybrid_rrf(item["question"])

    print("\n", question_id)
    print(
        results[
            ["document_id", "page_number", "score"]
        ].to_string(index=False)
    )

In [ ]:
def evaluate_custom_retriever(
    questions: list[dict],
    retriever,
) -> pd.DataFrame:
    """Evaluate a retriever that returns ranked chunk results."""

    evaluation_records = []

    for item in questions:
        if not item["answerable"]:
            continue

        gold_pages = {
            (document_id, int(page_number))
            for document_id, page_numbers in item["gold_pages"].items()
            for page_number in page_numbers
        }

        retrieved = retriever(
            query=item["question"],
            top_k=5,
        )

        retrieved_pages = list(
            zip(
                retrieved["document_id"],
                retrieved["page_number"].astype(int),
            )
        )

        record = {
            "question_id": item["question_id"],
            "category": item["category"],
            "gold_page_count": len(gold_pages),
        }

        for k in (1, 3, 5):
            top_k_pages = set(retrieved_pages[:k])
            matched_pages = gold_pages & top_k_pages

            record[f"recall_at_{k}"] = (
                len(matched_pages) / len(gold_pages)
            )
            record[f"hit_at_{k}"] = int(bool(matched_pages))

        evaluation_records.append(record)

    return pd.DataFrame(evaluation_records)

In [ ]:
hybrid_evaluation = evaluate_custom_retriever(
    questions=questions,
    retriever=retrieve_hybrid_rrf,
)

print("Mean metrics:")
print(hybrid_evaluation[metric_columns].mean().round(3))

print("\nQuestions without full Recall@5:")
print(
    hybrid_evaluation.loc[
        hybrid_evaluation["recall_at_5"] < 1,
        [
            "question_id",
            "category",
            "gold_page_count",
            "recall_at_5",
            "hit_at_5",
        ],
    ].to_string(index=False)
)

### Hybrid Evaluation Interpretation

The hybrid retriever produced the strongest development result: Recall@5 increased to 0.815 and Hit@5 to 0.889. It improved early ranking as well, with Recall@1 rising to 0.537.

This result does not mean the system is finished. One amendment question remained a complete miss, and the synthesis question retrieved only one of three required pages. The development set is also too small for a final performance claim. The hybrid configuration is selected for locked evaluation because it is the strongest tested development configuration, not because it is proven production quality.

Hybrid retrieval is the strongest development configuration so far, but it still has important amendment and cross-document synthesis failures and has not passed final evaluation.

In [ ]:
failed_item = next(
    question
    for question in questions
    if question["question_id"] == "DEV_AMEND_002"
)

failed_results = retrieve_hybrid_rrf(
    failed_item["question"],
    top_k=5,
)

result_display = failed_results[
    ["document_id", "page_number", "score", "chunk_text"]
].copy()

result_display["text_preview"] = (
    result_display["chunk_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 300)
)

print("Question:")
print(failed_item["question"])

print("\nGold pages:")
print(failed_item["gold_pages"])

print("\nRetrieved results:")
print(
    result_display[
        ["document_id", "page_number", "score", "text_preview"]
    ].to_string(index=False)
)

print("\nGold-page chunks:")
gold_chunks = revised_word_chunks.loc[
    (
        revised_word_chunks["document_id"]
        == "macpc_amendment_2024"
    )
    & (revised_word_chunks["page_number"] == 38),
    ["chunk_index", "chunk_text"],
]

for _, row in gold_chunks.iterrows():
    print(f"\nChunk {row['chunk_index']}:")
    print(row["chunk_text"])

In [ ]:
for page_number in [37, 38]:
    page_text = pages.loc[
        (
            pages["document_id"]
            == "macpc_amendment_2024"
        )
        & (pages["page_number"] == page_number),
        "text",
    ].iloc[0]

    print("\n" + "=" * 80)
    print(f"PAGE {page_number}")
    print(page_text)

In [ ]:
page_38_chunks = revised_word_chunks.loc[
    (
        revised_word_chunks["document_id"]
        == "macpc_amendment_2024"
    )
    & (revised_word_chunks["page_number"] == 38),
    ["chunk_id", "chunk_index", "chunk_text"],
]

answer_chunks = page_38_chunks.loc[
    page_38_chunks["chunk_text"].str.contains(
        "8A|two weeks",
        case=False,
        regex=True,
    )
]

print("Answer-bearing chunks:")
for _, row in answer_chunks.iterrows():
    print(f"\nChunk {row['chunk_index']}:")
    print(row["chunk_text"])

In [ ]:
question_text = failed_item["question"]

tfidf_all = retrieve_tfidf(
    query=question_text,
    chunks=revised_word_chunks,
    vectorizer=tfidf_140_vectorizer,
    chunk_matrix=tfidf_140_matrix,
    top_k=len(revised_word_chunks),
)

bge_all = retrieve_semantic(
    query=BGE_QUERY_INSTRUCTION + question_text,
    chunks=revised_word_chunks,
    model=bge_model,
    chunk_embeddings=bge_chunk_embeddings,
    top_k=len(revised_word_chunks),
)

for retriever_name, results in [
    ("TF-IDF", tfidf_all),
    ("BGE", bge_all),
]:
    matching = results.loc[
        results["chunk_id"].isin(answer_chunks["chunk_id"])
    ].copy()

    matching["rank"] = matching.index + 1

    print(f"\n{retriever_name} answer-chunk ranks:")
    print(
        matching[
            ["chunk_id", "rank", "score"]
        ].to_string(index=False)
    )

### Failure Analysis: Amendment Question

The answer-bearing paragraph 8A chunk ranked 2 with TF-IDF but 68 with BGE. Equal-weight RRF therefore weakened evidence that the lexical retriever found very strongly because the semantic retriever ranked it poorly.

This is a concrete fusion failure, not evidence that the gold page or chunk is missing. It also explains why blindly preferring semantic retrieval would be a mistake for exact legal identifiers such as `8A`. A later system may need rank-aware routing, stronger lexical treatment of legal references, or a different fusion design. These possibilities must be tested rather than hardcoded for one question.

In [ ]:
synthesis_item = next(
    question
    for question in questions
    if question["question_id"] == "DEV_SYNTH_001"
)

synthesis_query = synthesis_item["question"]

synthesis_tfidf_all = retrieve_tfidf(
    query=synthesis_query,
    chunks=revised_word_chunks,
    vectorizer=tfidf_140_vectorizer,
    chunk_matrix=tfidf_140_matrix,
    top_k=len(revised_word_chunks),
)

synthesis_bge_all = retrieve_semantic(
    query=BGE_QUERY_INSTRUCTION + synthesis_query,
    chunks=revised_word_chunks,
    model=bge_model,
    chunk_embeddings=bge_chunk_embeddings,
    top_k=len(revised_word_chunks),
)

synthesis_gold_pages = {
    (document_id, int(page_number))
    for document_id, page_numbers
    in synthesis_item["gold_pages"].items()
    for page_number in page_numbers
}

for retriever_name, results in [
    ("TF-IDF", synthesis_tfidf_all),
    ("BGE", synthesis_bge_all),
]:
    ranked = results.copy()
    ranked["rank"] = ranked.index + 1

    ranked["page_key"] = list(
        zip(
            ranked["document_id"],
            ranked["page_number"].astype(int),
        )
    )

    gold_rankings = (
        ranked.loc[ranked["page_key"].isin(synthesis_gold_pages)]
        .groupby(
            ["document_id", "page_number"],
            as_index=False,
        )["rank"]
        .min()
        .sort_values("rank")
    )

    print(f"\n{retriever_name} best gold-page ranks:")
    print(gold_rankings.to_string(index=False))

In [ ]:
report_subquery = (
    "How many flight cancellation complaints were reported "
    "in the second half of 2024?"
)

regulatory_subquery = (
    "What options does the principal Code provide to passengers "
    "affected by flight cancellation?"
)

for label, subquery in [
    ("Report subquery", report_subquery),
    ("Regulatory subquery", regulatory_subquery),
]:
    tfidf_results = retrieve_tfidf(
        query=subquery,
        chunks=revised_word_chunks,
        vectorizer=tfidf_140_vectorizer,
        chunk_matrix=tfidf_140_matrix,
        top_k=len(revised_word_chunks),
    )

    bge_results = retrieve_semantic(
        query=BGE_QUERY_INSTRUCTION + subquery,
        chunks=revised_word_chunks,
        model=bge_model,
        chunk_embeddings=bge_chunk_embeddings,
        top_k=len(revised_word_chunks),
    )

    if label == "Report subquery":
        allowed_documents = {"report_2h2024"}
    else:
        allowed_documents = {"macpc_principal_2016"}

    print(f"\n{label}")

    for retriever_name, results in [
        ("TF-IDF", tfidf_results),
        ("BGE", bge_results),
    ]:
        filtered = results.loc[
            results["document_id"].isin(allowed_documents)
        ].head(5)

        print(f"\n{retriever_name}:")
        print(
            filtered[
                ["document_id", "page_number", "score"]
            ].to_string(index=False)
        )

### Failure Analysis: Cross-Document Synthesis

The original synthesis query combined a report statistic with a regulatory-rights question. Both TF-IDF and BGE focused mainly on the report-related language, causing the principal Code evidence to rank poorly.

Manual query decomposition substantially improved retrieval. Report page 7 ranked first with BGE, while principal Code page 62 improved from rank 66 to rank 3. This confirms that mixed information needs were a major cause of failure.

Principal Code page 63 remained outside the top five because it continues a provision that begins on page 62. This exposes a separate page-boundary problem. Possible future solutions include adjacent-page expansion or section-aware chunks with citation spans.

Manual subqueries and hardcoded filters are diagnostic tools only. They should not be presented as a production solution without a general decomposition and routing method.

In [ ]:
weighted_rrf_evaluations = {}
weight_records = []

for tfidf_weight in [0.5, 0.6, 0.7, 0.8]:
    retriever = (
        lambda query, top_k, weight=tfidf_weight:
        retrieve_hybrid_rrf(
            query=query,
            top_k=top_k,
            tfidf_weight=weight,
        )
    )

    results = evaluate_custom_retriever(
        questions=questions,
        retriever=retriever,
    )

    weighted_rrf_evaluations[tfidf_weight] = results

    means = results[metric_columns].mean()

    weight_records.append(
        {
            "tfidf_weight": tfidf_weight,
            "bge_weight": 1 - tfidf_weight,
            "recall_at_1": means["recall_at_1"],
            "recall_at_3": means["recall_at_3"],
            "recall_at_5": means["recall_at_5"],
            "hit_at_5": means["hit_at_5"],
        }
    )

weight_comparison = pd.DataFrame(weight_records)

print(weight_comparison.round(3).to_string(index=False))

### Weighted RRF Interpretation

TF-IDF weights from 0.5 to 0.8 produced identical Recall@K and Hit@5 metrics. Within this tested range, weighting did not change whether gold pages entered the evaluated rank positions.

Further weight searching was stopped to avoid tuning the system specifically around known development failures. Equal weighting was retained because it is the simplest configuration with the same measured performance.

The remaining failures require structural solutions, such as query decomposition, metadata routing, or adjacent-page expansion, rather than minor fusion-weight adjustments.

In [ ]:
print(pages.columns.tolist())

### Metadata Interpretation

The page dataset contains more than text and page numbers. It preserves document type, authority, Gazette number, publication and effective dates, reporting periods, source URL, and retrieval eligibility.

These fields create options that similarity search alone cannot provide. For example, a future retriever could restrict a query to the requested reporting period, Gazette authority, or effective date. Printing the columns here confirms that the metadata required for later filtering and routing was preserved during ingestion.

## Reproducible Selected Configuration

The selected development configuration is rebuilt using tested package modules rather than notebook-only functions:

- 140-word chunks with 25-word overlap
- TF-IDF lexical retrieval
- BGE small English semantic retrieval
- BGE query instruction
- Equal-weight Reciprocal Rank Fusion
- Candidate depth of 20
- Final retrieval depth of 5

This integration check ensures that the reusable implementation reproduces the experimental result.

In [ ]:
import importlib

import aviation_rag.evaluation_data
import aviation_rag.retrieval

importlib.reload(aviation_rag.evaluation_data)
importlib.reload(aviation_rag.retrieval)

In [ ]:
from aviation_rag.evaluation_data import evaluate_retriever
from aviation_rag.retrieval import (
    HybridRetriever,
    SemanticRetriever,
    TfidfRetriever,
)


stable_tfidf_retriever = TfidfRetriever(
    revised_word_chunks
)

stable_bge_retriever = SemanticRetriever(
    chunks=revised_word_chunks,
    model=bge_model,
    query_prefix=BGE_QUERY_INSTRUCTION,
)

stable_hybrid_retriever = HybridRetriever(
    lexical_retriever=stable_tfidf_retriever,
    semantic_retriever=stable_bge_retriever,
    candidate_k=20,
    rrf_constant=60,
    lexical_weight=0.5,
)

stable_hybrid_evaluation = evaluate_retriever(
    questions=questions,
    retriever=stable_hybrid_retriever,
)

print(
    stable_hybrid_evaluation[
        metric_columns
    ].mean().round(3)
)

### Integration Check Interpretation

The tested package implementation reproduced the experimental hybrid metrics exactly. This confirms that the selected chunking, lexical retrieval, semantic retrieval, query instruction, and RRF configuration were transferred correctly from notebook exploration into reusable code.

The notebook remains the record of experimentation and interpretation, while the package modules now contain the stable implementation.